In [127]:
%load_ext autoreload
%autoreload 2
import torch 
import torchvision
from torchvision import transforms
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from physics_utils import *
from dataset_dataloader import *
import yaml
from torch.utils.data import WeightedRandomSampler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [128]:
#temp_train_ds=DatasetMaker('../data/train_data.csv',transforms=transforms.Compose([FftTransform(),transforms.ToTensor()]))
#all=[temp_train_ds[i][0] for i in range(len(temp_train_ds))]
#a=torch.stack(all,dim=0)
#train_mean=a[:,0,:,:].mean().item()
#train_std=a[:,0,:,:].std().item()  ## add these stats to the config file for further use

In [129]:
config_path = '../src/config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

In [130]:
#config['stats']['train_mean'] = round(train_mean, 4)
#config['stats']['train_std'] = round(train_std, 4)
counts_dict = config['train_class_count']
counts = torch.tensor([counts_dict[i] for i in range(len(counts_dict))], dtype=torch.float32)
class_weights = 1.0 / counts
print(f"Calculated Weights: {class_weights}")
config['class_weight']=class_weights.tolist()
config['batch_size']=32
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print("YAML updated")

Calculated Weights: tensor([0.0008, 0.0042, 0.0118, 0.0017])
YAML updated


In [131]:
train_transform = transforms.Compose([FftTransform(width=config['fft_params']['notch_width'], notch_depth=config['fft_params']['notch_depth'],
                                                   apply_bilateral=config['fft_params']['apply_bilateral']),
                                      transforms.RandomHorizontalFlip(p=0.5),
                                      transforms.RandomVerticalFlip(p=0.5),
                    # rotations (90 deg increase) to not destroy the fft transform
                                      transforms.RandomChoice([
                                      transforms.RandomRotation((0, 0)),
                                      transforms.RandomRotation((90, 90)),
                                      transforms.RandomRotation((180, 180)),
                                      transforms.RandomRotation((270, 270))]),
                                      transforms.ToTensor(),
                                      transforms.Normalize(mean=[config['stats']['train_mean']], std=[config['stats']['train_std']])])

val_transform =transforms.Compose([FftTransform(width=config['fft_params']['notch_width'], notch_depth=config['fft_params']['notch_depth'],
                                                    apply_bilateral=config['fft_params']['apply_bilateral']),
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean=[config['stats']['train_mean']], std=[config['stats']['train_std']])])

In [132]:
train_ds=DatasetMaker(data_csv_path=config['data_path']['train'],transforms=train_transform)
val_ds=DatasetMaker(data_csv_path=config['data_path']['val'],transforms=val_transform)
test_ds=DatasetMaker(data_csv_path=config['data_path']['test'],transforms=val_transform)

**we weight each image label so that we fix the imbalanced labels issue in the dataset where we have majority call of label 0**

In [133]:
# Create a weight for EACH sample based on its class label
sample_weights = []
for idx in range(len(train_ds)):
    label = train_ds.data['label'][idx]
    sample_weights.append(config['class_weight'][label])

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_ds),
    replacement=True
)

In [134]:
train_dl=DataLoader(train_ds,batch_size=config['batch_size'],sampler=sampler)
test_dl=DataLoader(test_ds,batch_size=config['batch_size'])
val_dl=DataLoader(val_ds,batch_size=config['batch_size'])